# Reinforcement Learning (From Scratch) — A Toy Gridworld

_Generated: 2025-10-22T19:55:07.187977Z_

This notebook implements **tabular model-free RL** from first principles using only NumPy. We use a small **Gridworld** with stochastic wind, immovable walls, a **goal** (+10), and **pits** (−10) and step cost (−1). We implement **Q‑Learning** (off‑policy) and optionally **SARSA** (on‑policy), with a **dynamic programming (value iteration)** baseline for comparison.

**You’ll get:** value heatmaps, greedy policy arrows, learning curves, and a rollout animation of the learned policy.

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess, math
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib imageio

## 1) Imports & Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

## 2) Gridworld Environment (first principles)

In [ ]:
from dataclasses import dataclass

UP, RIGHT, DOWN, LEFT = 0, 1, 2, 3
ACTIONS = np.array([UP, RIGHT, DOWN, LEFT])
ARROWS  = {UP:"↑", RIGHT:"→", DOWN:"↓", LEFT:"←"}

@dataclass
class GWConfig:
    H:int=5; W:int=7
    step_cost:float=-1.0
    goal_reward:float=10.0
    pit_reward:float=-10.0
    p_slip:float=0.15
    seed:int=SEED

CFG = GWConfig()

EMPTY, WALL, GOAL, PIT = 0, 1, 2, 3

def make_world():
    M = np.zeros((CFG.H, CFG.W), dtype=int)
    M[0,:] = WALL; M[-1,:] = WALL; M[:,0] = WALL; M[:,-1] = WALL
    M[2,2:5] = WALL
    M[1,4] = WALL
    M[1,5] = GOAL
    M[3,3] = PIT
    start = (CFG.H-2, 1)
    return M, start

WORLD, START = make_world()

def is_terminal(cell):
    r,c = cell
    t = WORLD[r,c]
    return t==GOAL or t==PIT

def step(state, action, rng=rng):
    r,c = state
    a = action
    if rng.random() < CFG.p_slip:
        a = (action + (1 if rng.random()<0.5 else -1)) % 4
    dr = [ -1,  0,  1,  0 ][a]
    dc = [  0,  1,  0, -1 ][a]
    nr, nc = r + dr, c + dc
    if WORLD[nr, nc] == WALL:
        nr, nc = r, c
    rew = CFG.step_cost
    tile = WORLD[nr, nc]
    done = False
    if tile == GOAL:
        rew += CFG.goal_reward
        done = True
    elif tile == PIT:
        rew += CFG.pit_reward
        done = True
    return (nr, nc), rew, done

def reset():
    return START

## 3) State Indexing, Masks, and Visualization Helpers

In [ ]:
S = CFG.H * CFG.W
A = 4

def rc_to_s(r,c): return r*CFG.W + c
def s_to_rc(s):   return (s//CFG.W, s%CFG.W)

valid_mask = np.array([WORLD[s_to_rc(s)] != WALL for s in range(S)], dtype=bool)
state_list = np.where(valid_mask)[0]
nS = len(state_list)

def plot_value_and_policy(V, Pi, title="Value & Greedy Policy"):
    gridV = np.full((CFG.H, CFG.W), np.nan)
    gridP = np.full((CFG.H, CFG.W), None, dtype=object)
    for s in range(S):
        if not valid_mask[s]: continue
        r,c = s_to_rc(s)
        gridV[r,c] = V[s]
        gridP[r,c] = {0:"↑",1:"→",2:"↓",3:"←"}[int(Pi[s])] if Pi is not None else ""
    fig, ax = plt.subplots(1,1, figsize=(6,4.5))
    im = ax.imshow(gridV, cmap="viridis")
    for r in range(CFG.H):
        for c in range(CFG.W):
            if WORLD[r,c] == 1:
                ax.add_patch(plt.Rectangle((c-0.5,r-0.5),1,1,color="black"))
            elif WORLD[r,c] == 2:
                ax.text(c, r, "G", ha="center", va="center", color="white", fontsize=12, fontweight="bold")
            elif WORLD[r,c] == 3:
                ax.text(c, r, "X", ha="center", va="center", color="white", fontsize=12, fontweight="bold")
            elif not np.isnan(gridV[r,c]):
                ax.text(c, r, gridP[r,c], ha="center", va="center", color="white", fontsize=12)
    ax.set_title(title); plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

## 4) Dynamic Programming Baseline — Value Iteration (reference)

In [ ]:
def value_iteration(gamma=0.98, theta=1e-5, max_iter=10000):
    V = np.zeros(S, dtype=float)
    for it in range(max_iter):
        delta = 0.0
        for s in range(S):
            if not valid_mask[s]: continue
            r,c = s_to_rc(s)
            if (WORLD[r,c] in (2,3)):
                V[s] = 0.0
                continue
            vals = []
            for a in range(4):
                # expected over slip
                probs = {a: 1.0- CFG.p_slip, (a-1)%4: CFG.p_slip/2.0, (a+1)%4: CFG.p_slip/2.0}
                ex = 0.0
                for aa, pa in probs.items():
                    (nr,nc), rew, done = step((r,c), aa, rng=np.random.default_rng(9999))
                    ns = rc_to_s(nr,nc)
                    ex += pa * (rew + (0.0 if done else gamma*V[ns]))
                vals.append(ex)
            v_new = np.max(vals)
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta: break
    Pi = np.zeros(S, dtype=int)
    for s in range(S):
        if not valid_mask[s]: continue
        r,c = s_to_rc(s)
        if (WORLD[r,c] in (2,3)):
            Pi[s] = 0; continue
        vals = []
        for a in range(4):
            probs = {a: 1.0- CFG.p_slip, (a-1)%4: CFG.p_slip/2.0, (a+1)%4: CFG.p_slip/2.0}
            ex = 0.0
            for aa, pa in probs.items():
                (nr,nc), rew, done = step((r,c), aa, rng=np.random.default_rng(9999))
                ns = rc_to_s(nr,nc)
                ex += pa * (rew + (0.0 if done else gamma*V[ns]))
            vals.append(ex)
        Pi[s] = int(np.argmax(vals))
    return V, Pi

V_star, Pi_star = value_iteration(gamma=0.98)
plot_value_and_policy(V_star, Pi_star, title="Value Iteration (reference) — V* and greedy π*")

## 5) Tabular Q‑Learning (off‑policy, ε‑greedy)

In [ ]:
from dataclasses import dataclass
@dataclass
class QLConfig:
    episodes:int=2000
    alpha:float=0.2
    gamma:float=0.98
    eps_start:float=1.0
    eps_end:float=0.05
    eps_decay:float=0.995
    max_steps:int=200
QLCFG = QLConfig()

def epsilon_greedy(q_row, eps):
    if rng.random() < eps:
        return int(rng.integers(0, 4))
    return int(np.argmax(q_row))

def q_learning(cfg=QLCFG):
    Q = np.zeros((S, 4), dtype=float)
    eps = cfg.eps_start
    returns = []
    for ep in range(cfg.episodes):
        s_rc = reset()
        s = rc_to_s(*s_rc)
        G = 0.0
        for t in range(cfg.max_steps):
            a = epsilon_greedy(Q[s], eps) if (WORLD[s_to_rc(s)]!=1) else int(rng.integers(0,4))
            (nr,nc), r, done = step(s_rc, a, rng=rng)
            ns = rc_to_s(nr,nc)
            td_target = r + (0.0 if done else cfg.gamma * np.max(Q[ns]))
            Q[s,a] += cfg.alpha * (td_target - Q[s,a])
            s_rc = (nr,nc); s = ns; G += r
            if done: break
        eps = max(cfg.eps_end, eps*cfg.eps_decay)
        returns.append(G)
        if (ep+1) % 200 == 0:
            print(f"Ep {ep+1:4d} | return={G:6.2f} | eps={eps:.3f}")
    return Q, np.array(returns)

Q, ret_hist = q_learning(QLCFG)
V_hat = np.max(Q, axis=1)
Pi_hat = np.argmax(Q, axis=1)
plot_value_and_policy(V_hat, Pi_hat, title="Q‑Learning — V^ and greedy π(Q)")

fig = plt.figure(figsize=(6,4))
plt.plot(ret_hist, alpha=0.7)
if len(ret_hist)>50:
    mv = np.convolve(ret_hist, np.ones(50)/50, mode='valid')
    plt.plot(np.arange(len(mv))+49, mv, label="moving avg (50)")
plt.xlabel("Episode"); plt.ylabel("Return"); plt.title("Q‑Learning: episodic return")
plt.legend(); plt.tight_layout(); plt.show()

## 6) (Optional) SARSA (on‑policy)

In [ ]:
def sarsa(cfg=QLCFG):
    Q = np.zeros((S, 4), dtype=float)
    eps = cfg.eps_start
    returns = []
    for ep in range(cfg.episodes):
        s_rc = reset(); s = rc_to_s(*s_rc)
        a = epsilon_greedy(Q[s], eps)
        G = 0.0
        for t in range(cfg.max_steps):
            (nr,nc), r, done = step(s_rc, a, rng=rng); ns = rc_to_s(nr,nc)
            an = epsilon_greedy(Q[ns], eps) if not done else 0
            td_target = r + (0.0 if done else cfg.gamma * Q[ns, an])
            Q[s,a] += cfg.alpha * (td_target - Q[s,a])
            s_rc=(nr,nc); s=ns; a=an; G+=r
            if done: break
        eps = max(cfg.eps_end, eps*cfg.eps_decay)
        returns.append(G)
    return Q, np.array(returns)

# To try SARSA:
# Qs, ret_sarsa = sarsa(QLCFG)
# plot_value_and_policy(np.max(Qs,1), np.argmax(Qs,1), title="SARSA — V^ and greedy π(Q)")
# plt.figure(figsize=(6,4)); plt.plot(ret_sarsa); plt.title("SARSA: episodic return"); plt.show()

## 7) Rollout Rendering (GIF)

In [ ]:
def render_frame(state):
    r,c = state
    H,W = WORLD.shape
    img = np.zeros((H, W, 3), dtype=np.uint8)
    img[:,:,:] = [30,30,45]
    for rr in range(H):
        for cc in range(W):
            if WORLD[rr,cc]==1: img[rr,cc]=[0,0,0]
            elif WORLD[rr,cc]==2: img[rr,cc]=[30,160,60]
            elif WORLD[rr,cc]==3: img[rr,cc]=[180,40,40]
            else: img[rr,cc]=[70,70,95]
    img[r,c] = [230, 210, 60]
    scale=40
    return np.kron(img, np.ones((scale,scale,1), dtype=np.uint8))

def rollout_greedy(Q, max_steps=200):
    s = reset(); frames=[]; total=0.0
    for t in range(max_steps):
        frames.append(render_frame(s))
        if is_terminal(s): break
        sr = rc_to_s(*s)
        a = int(np.argmax(Q[sr]))
        s, r, done = step(s, a, rng=rng); total += r
        if done:
            frames.append(render_frame(s)); break
    return total, frames

G_eval, frames = rollout_greedy(Q, max_steps=200)
print("Greedy rollout return:", G_eval)

import os
os.makedirs("artifacts", exist_ok=True)
gif_path = "artifacts/gridworld_qlearning_rollout.gif"
if frames:
    imageio.mimsave(gif_path, frames, duration=1/6.0)
    print("Saved animation:", gif_path)

## 8) Save Artifacts & Download

In [ ]:
import os, json as _json, numpy as _np, shutil
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/qlearning_results.npz",
         Q=Q, returns=ret_hist, V_hat=np.max(Q,axis=1), Pi_hat=np.argmax(Q,axis=1),
         V_star=V_star, Pi_star=Pi_star)

with open("artifacts/world_spec.json","w") as f:
    _json.dump({"H": CFG.H, "W": CFG.W, "p_slip": CFG.p_slip,
                "rewards":{"step":CFG.step_cost, "goal":CFG.goal_reward, "pit":CFG.pit_reward}}, f, indent=2)

try:
    from google.colab import files  # type: ignore
    shutil.make_archive('artifacts','zip','artifacts'); files.download('artifacts.zip')
except Exception as e:
    print("Colab download helper not available here:", e)

## 9) Exercises & Extensions

- Switch to **SARSA** and compare curves.
- Try different `ε` and `α` schedules; optimistic starts.
- Increase slip `p_slip` or move the pit/goal; observe robustness.
- Implement **Expected SARSA** or **Double Q‑Learning**.
- Replace the table with a linear function approximator to scale up.
